# 编排模式： 监管者、群、分层

2026年的框架中反复出现四种编排框架：supervisor-worker，swarm/peer-to-peer，hierachical，debate。Anthropic 的指引：根据你的需求构建正确的系统。从简单的开始，只在单个agent加上5种工作流模式不够用的时候才去加拓扑。

## 基本概念

### Supervisor —— Worker

- 中心路由LLM将任务下分给专家agents。
- 决定：循环回自己、分派给专家、终止。
- 专家之间互相不沟通，所有的路由都需要通过supervisor。

框架：LangGraph————`create_supervisor`； Anthropic ———— orchestrator-workers；CrewAI———— Hierachical Process。

2026年LangChain 的建议： 通过直接的工具调用做监管。这会给你更好的上下文工程共知————你明确定义哪些内容会被各个专家看见。

### SWARM / PEER-TO——PEER

- agents之间通过共享的工具层直接进行交接
- 没有中心的路由
- 比supervisor 更低的延迟
- 过程难以推理

框架：LangGraph ———— swarm topology； OpenAI Agents SDK ———— handoffs

### Hierachical

- Supervisor 管 sub-Supervisor 再管 workers。
- LangGraph 中使用嵌套的子图；CrewAI中使用嵌套的crews。
- 以操作复杂性的代价推广到大人口的agents。

什么时候需要：当单个supervisor的上下文窗口预算不能够承载所有的专家描述时。

### Debate

- 并行提案 + 迭代交叉批判
- 并不是一种编排模式，更倾向于验证，但也作为框架中拓扑选择的一种。

### Anthropic 的指引

LLM空间中的成功不来自构建最复杂的系统，而是根据你的实际需求构建正确的系统。决策顺序：
- 单个agent + 工作流模式。  从这里开始
- supervisor-worker。  当你有2-4个专家的时候
- swarm。  当延迟比推理清晰度更重要的时候
- hierachical。 仅当supervisor上下文预算不够用的时候
- debate。  当准确度比开销更重要的时候

### 什么时候模式失效

- 先考虑拓扑。   先于识别哪些问题需要多agents解决就“我们需要多个agents”
- swarm中弹跳传递。  A到B，B又递回给A。加上计数限制。
- 虚假hierarchy。  压缩掉无意义的层。

# 开始编码

对应本章核心：**Supervisor（中心路由，专家互不通信）**、**Swarm（点对点交接 + 跳数上限）**、**Hierarchical（嵌套子监管）**、**按需求选拓扑（先单 agent，再加层）**。  
Debate 上节已实现，本节当验证拓扑点到为止。先用脚本化运行时跑通三种编排与失败模式；再用 **LangGraph + DeepSeek** 做生产 Supervisor（工具调用监管）。不硬凑 PyTorch。


## 1. 教学玩具：三种编排运行时

- **Supervisor**：只经中心；边只能是 `sup↔worker`。
- **Swarm**：当前持有者 `handoff`；`hops` 超限则停（防 A↔B 弹跳）。
- **Hierarchical**：top → mid → worker；专家描述超上下文预算才拆层。


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Callable, Literal

Pattern = Literal["single", "supervisor", "swarm", "hierarchical", "debate"]


@dataclass
class Hop:
    """一次编排跳转。"""

    src: str
    dst: str
    reason: str


@dataclass
class RunLog:
    """一场编排的轨迹。"""

    pattern: Pattern
    hops: list[Hop] = field(default_factory=list)
    final: str = ""
    stopped: str = ""


def pick_pattern(
    *,
    n_experts: int,
    latency_over_clarity: bool = False,
    supervisor_ctx_overflow: bool = False,
    accuracy_over_cost: bool = False,
) -> Pattern:
    """
    Anthropic 决策顺序：单 agent → supervisor → swarm → hierarchical → debate。

    Returns:
        pattern: 推荐拓扑。
    """
    if n_experts <= 1:
        return "single"
    if accuracy_over_cost:
        return "debate"
    if supervisor_ctx_overflow:
        return "hierarchical"
    if latency_over_clarity:
        return "swarm"
    if 2 <= n_experts <= 4:
        return "supervisor"
    return "hierarchical"


class SupervisorRuntime:
    """中心路由：专家互不通信。"""

    def __init__(self, experts: dict[str, Callable[[str], str]], router: Callable[[str], str]) -> None:
        self.experts = experts
        self.router = router

    def run(self, task: str, *, max_steps: int = 6) -> RunLog:
        """
        Args:
            task: 用户任务。
            max_steps: 监管循环上限。

        Returns:
            log: hops 只含 supervisor↔worker 或 END。
        """
        log = RunLog(pattern="supervisor")
        current = task
        for _ in range(max_steps):
            dest = self.router(current)
            if dest == "END":
                log.hops.append(Hop("supervisor", "END", "terminate"))
                log.final = current
                log.stopped = "ok"
                return log
            if dest not in self.experts:
                log.stopped = f"unknown_expert:{dest}"
                return log
            log.hops.append(Hop("supervisor", dest, "dispatch"))
            current = self.experts[dest](current)
            log.hops.append(Hop(dest, "supervisor", "return"))
        log.final = current
        log.stopped = "max_steps"
        return log

    @staticmethod
    def worker_talked(log: RunLog) -> bool:
        """专家之间是否出现过边（违反 supervisor 不互通）。"""
        names = {h.src for h in log.hops} | {h.dst for h in log.hops}
        workers = names - {"supervisor", "END"}
        return any(h.src in workers and h.dst in workers for h in log.hops)


class SwarmRuntime:
    """点对点交接：无中心；用 hop 上限防弹跳。"""

    def __init__(
        self,
        agents: dict[str, Callable[[str], tuple[str, str | None]]],
        *,
        start: str,
        max_hops: int = 4,
    ) -> None:
        self.agents = agents
        self.start = start
        self.max_hops = max_hops

    def run(self, task: str) -> RunLog:
        """
        每个 agent 返回 ``(output, next_agent|None)``。

        Returns:
            log: 含 bounce 检测。
        """
        log = RunLog(pattern="swarm")
        current_id = self.start
        payload = task
        seen_pair: set[tuple[str, str]] = set()
        for _ in range(self.max_hops + 1):
            if current_id not in self.agents:
                log.stopped = f"unknown:{current_id}"
                return log
            out, nxt = self.agents[current_id](payload)
            payload = out
            if nxt is None:
                log.final = out
                log.stopped = "ok"
                return log
            pair = (current_id, nxt)
            if pair in seen_pair:
                log.hops.append(Hop(current_id, nxt, "bounce"))
                log.final = out
                log.stopped = "bounce"
                return log
            seen_pair.add(pair)
            if len(log.hops) >= self.max_hops:
                log.hops.append(Hop(current_id, nxt, "hop_limit"))
                log.final = out
                log.stopped = "hop_limit"
                return log
            log.hops.append(Hop(current_id, nxt, "handoff"))
            current_id = nxt
        log.final = payload
        log.stopped = "hop_limit"
        return log


class HierarchicalRuntime:
    """嵌套监管：top → mid → worker。"""

    def __init__(
        self,
        *,
        top_router: Callable[[str], str],
        mids: dict[str, Callable[[str], str]],
        workers: dict[str, Callable[[str], str]],
        mid_to_workers: dict[str, list[str]],
        ctx_chars: int,
        budget: int,
    ) -> None:
        self.top_router = top_router
        self.mids = mids
        self.workers = workers
        self.mid_to_workers = mid_to_workers
        self.ctx_chars = ctx_chars
        self.budget = budget

    def needs_hierarchy(self, expert_blurbs: list[str]) -> bool:
        """单个 supervisor 塞不下所有专家描述时才拆层。"""
        return sum(len(b) for b in expert_blurbs) > self.budget

    def run(self, task: str) -> RunLog:
        """
        Returns:
            log: 两层路由 hops。
        """
        log = RunLog(pattern="hierarchical")
        mid = self.top_router(task)
        log.hops.append(Hop("top", mid, "dispatch"))
        mid_out = self.mids[mid](task)
        log.hops.append(Hop(mid, "top", "ack"))
        # mid 选自己的 worker
        cands = self.mid_to_workers[mid]
        worker = cands[0]
        for w in cands:
            if w in mid_out or w in task:
                worker = w
                break
        log.hops.append(Hop(mid, worker, "dispatch"))
        log.final = self.workers[worker](task)
        log.hops.append(Hop(worker, mid, "return"))
        log.stopped = "ok"
        return log


print("orchestration toys ready | supervisor + swarm + hierarchical")


## 2. 玩具示例：中心边、弹跳上限、拆层预算


In [ ]:
def demo_orchestration_toy() -> None:
    """断言三种拓扑形状与失败模式。"""
    assert pick_pattern(n_experts=1) == "single"
    assert pick_pattern(n_experts=3) == "supervisor"
    assert pick_pattern(n_experts=3, latency_over_clarity=True) == "swarm"
    assert pick_pattern(n_experts=6, supervisor_ctx_overflow=True) == "hierarchical"
    assert pick_pattern(n_experts=3, accuracy_over_cost=True) == "debate"
    print("pick_pattern order ok")

    def research(t: str) -> str:
        return f"facts:{t}"

    def draft(t: str) -> str:
        return f"draft:{t}"

    calls = {"n": 0}

    def router(t: str) -> str:
        calls["n"] += 1
        if calls["n"] == 1:
            return "research"
        if calls["n"] == 2:
            return "draft"
        return "END"

    sup = SupervisorRuntime({"research": research, "draft": draft}, router)
    slog = sup.run("q")
    assert slog.final.startswith("draft:")
    assert not SupervisorRuntime.worker_talked(slog)
    assert all(
        {h.src, h.dst} & {"supervisor", "END"}
        for h in slog.hops
    )
    print("supervisor: experts never talk ok")

    # Swarm 弹跳 A→B→A，第二圈被拦
    def a(t: str) -> tuple[str, str | None]:
        return (t + "/A", "B")

    def b(t: str) -> tuple[str, str | None]:
        return (t + "/B", "A")

    sw = SwarmRuntime({"A": a, "B": b}, start="A", max_hops=3)
    blog = sw.run("q")
    assert blog.stopped in {"bounce", "hop_limit"}
    assert any(h.reason in {"bounce", "hop_limit"} for h in blog.hops)
    print(f"swarm bounce/limit -> {blog.stopped}")

    # 无弹跳：A→B→END
    def a2(t: str) -> tuple[str, str | None]:
        return (t + "/A", "B")

    def b2(t: str) -> tuple[str, str | None]:
        return (t + "/B", None)

    sw2 = SwarmRuntime({"A": a2, "B": b2}, start="A", max_hops=4)
    ok = sw2.run("q")
    assert ok.stopped == "ok" and ok.final.endswith("/B")
    print("swarm clean handoff ok")

    blurbs = ["research " * 20, "draft " * 20, "code " * 20]
    hier = HierarchicalRuntime(
        top_router=lambda t: "mid_write" if "写" in t or "draft" in t else "mid_know",
        mids={
            "mid_know": lambda t: "research",
            "mid_write": lambda t: "draft",
        },
        workers={"research": research, "draft": draft, "code": lambda t: t},
        mid_to_workers={"mid_know": ["research"], "mid_write": ["draft", "code"]},
        ctx_chars=sum(len(b) for b in blurbs),
        budget=80,
    )
    assert hier.needs_hierarchy(blurbs) is True
    hlog = hier.run("写一份摘要")
    assert hlog.hops[0].src == "top"
    assert any(h.src.startswith("mid_") for h in hlog.hops)
    assert hlog.final.startswith("draft:")
    print("hierarchical two-level dispatch ok")
    print("TOY DEMO OK")


demo_orchestration_toy()


## 3. 生产级：LangGraph Supervisor + DeepSeek

监管者用工具调用分派专家（LangChain 2026 建议：工具 = 专家，上下文你自己裁）。专家互不看见对方轨迹。需 `DEEPSEEK_API_KEY`。


In [ ]:
import json
import os
import sys
from pathlib import Path
from typing import Any, Literal

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"


class OrchState(TypedDict, total=False):
    """Supervisor 图状态。"""

    task: str
    route: str
    research: str
    draft: str
    hops: list[dict[str, str]]
    final: str
    step: int


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def _invoke(prompt: str) -> str:
    return str(get_llm().invoke(prompt).content).strip()


def supervisor_node(state: OrchState) -> dict[str, Any]:
    """
    中心路由：只看 task + 已有专家摘要，决定 research / draft / END。
    """
    task = state["task"]
    research = state.get("research") or ""
    draft = state.get("draft") or ""
    step = int(state.get("step") or 0)
    hops = list(state.get("hops") or [])
    prompt = (
        "You are a supervisor. Reply with EXACTLY one token: research | draft | END.\n"
        "Use research first if facts are missing. Use draft if facts exist but no writeup. END if draft exists.\n"
        f"Task: {task}\n"
        f"Research so far: {research[:400] or '(none)'}\n"
        f"Draft so far: {draft[:400] or '(none)'}\n"
        "Route:"
    )
    # 硬门控：有成稿或步数到顶必须停，避免监管者空转
    if draft or step >= 4:
        hops.append({"src": "supervisor", "dst": "END", "reason": "terminate"})
        return {"route": "END", "hops": hops, "step": step + 1, "final": draft or research or task}
    raw = _invoke(prompt).split()[0].strip("`. ").lower()
    if raw not in {"research", "draft", "end"}:
        raw = "draft" if research else "research"
    if raw == "draft" and not research:
        raw = "research"
    if raw == "end":
        hops.append({"src": "supervisor", "dst": "END", "reason": "terminate"})
        return {"route": "END", "hops": hops, "step": step + 1, "final": draft or research or task}
    hops.append({"src": "supervisor", "dst": raw, "reason": "dispatch"})
    return {"route": raw, "hops": hops, "step": step + 1}


def research_node(state: OrchState) -> dict[str, Any]:
    """专家：只看见 task，看不见 draft（监管裁上下文）。"""
    task = state["task"]
    out = _invoke(
        "You are the research expert. List 3 short factual bullets in Chinese for:\n"
        f"{task}\nBullets:"
    )
    hops = list(state.get("hops") or [])
    hops.append({"src": "research", "dst": "supervisor", "reason": "return"})
    return {"research": out, "hops": hops, "route": "supervisor"}


def draft_node(state: OrchState) -> dict[str, Any]:
    """专家：只看见 task + research，不看见其他专家闲聊。"""
    task = state["task"]
    research = state.get("research") or ""
    out = _invoke(
        "You are the draft expert. Write <=40 Chinese characters.\n"
        f"Task: {task}\nResearch:\n{research}\nDraft:"
    )
    hops = list(state.get("hops") or [])
    hops.append({"src": "draft", "dst": "supervisor", "reason": "return"})
    return {"draft": out, "final": out, "hops": hops, "route": "supervisor"}


def _route_sup(state: OrchState) -> str:
    r = (state.get("route") or "supervisor").lower()
    if r == "research":
        return "research"
    if r == "draft":
        return "draft"
    if r == "end":
        return "END"
    return "supervisor"


def build_supervisor_graph() -> Any:
    """
    Returns:
        graph: supervisor 中心图。
    """
    g = StateGraph(OrchState)
    g.add_node("supervisor", supervisor_node)
    g.add_node("research", research_node)
    g.add_node("draft", draft_node)
    g.add_edge(START, "supervisor")
    g.add_conditional_edges(
        "supervisor",
        _route_sup,
        {"research": "research", "draft": "draft", "END": END, "supervisor": "supervisor"},
    )
    g.add_edge("research", "supervisor")
    g.add_edge("draft", "supervisor")
    return g.compile()


SUP_GRAPH = build_supervisor_graph()


def run_supervisor_impl(task: str) -> str:
    """
    Returns:
        json: final + hops（可断言专家互不直连）。
    """
    out = SUP_GRAPH.invoke(
        {"task": task, "hops": [], "step": 0, "route": "supervisor"},
        config={"recursion_limit": 12},
    )
    hops = out.get("hops") or []
    workers = {"research", "draft"}
    leaked = [h for h in hops if h["src"] in workers and h["dst"] in workers]
    return json.dumps(
        {
            "final": out.get("final") or out.get("draft") or "",
            "research": out.get("research") or "",
            "hops": hops,
            "worker_leak": leaked,
        },
        ensure_ascii=False,
    )


class TaskArgs(BaseModel):
    task: str


class PickArgs(BaseModel):
    n_experts: int = 3
    latency_over_clarity: bool = False
    supervisor_ctx_overflow: bool = False
    accuracy_over_cost: bool = False


def build_control_tools() -> list[StructuredTool]:
    def _run(**kwargs: Any) -> str:
        return run_supervisor_impl(TaskArgs(**kwargs).task)

    def _pick(**kwargs: Any) -> str:
        a = PickArgs(**kwargs)
        p = pick_pattern(
            n_experts=a.n_experts,
            latency_over_clarity=a.latency_over_clarity,
            supervisor_ctx_overflow=a.supervisor_ctx_overflow,
            accuracy_over_cost=a.accuracy_over_cost,
        )
        return json.dumps({"pattern": p}, ensure_ascii=False)

    return [
        StructuredTool.from_function(
            name="run_supervisor",
            description="Run supervisor-worker graph (research/draft experts).",
            func=_run,
            args_schema=TaskArgs,
        ),
        StructuredTool.from_function(
            name="pick_pattern",
            description="Recommend orchestration pattern from Anthropic order.",
            func=_pick,
            args_schema=PickArgs,
        ),
    ]


CONTROL_TOOLS = build_control_tools()


def build_control_agent() -> Any:
    system = (
        "You operate an orchestration control plane.\n"
        "Use pick_pattern to choose topology; run_supervisor for 2-4 expert tasks.\n"
        "Explain that experts do not talk to each other. Chinese."
    )
    return create_agent(get_llm(), CONTROL_TOOLS, system_prompt=system)


def format_agent_messages(messages: list[BaseMessage]) -> str:
    lines: list[str] = []
    for m in messages:
        if isinstance(m, HumanMessage):
            lines.append(f"USER: {m.content}")
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    lines.append(f"ACTION: {tc['name']}({tc.get('args') or {}})")
            if m.content:
                lines.append(f"ASSISTANT: {m.content}")
        elif isinstance(m, ToolMessage):
            content = m.content if len(str(m.content)) < 600 else str(m.content)[:600] + "..."
            lines.append(f"OBS[{m.name}]: {content}")
    return "\n".join(lines)


print(f"LangGraph supervisor ready | {MODEL}")


## 4. 生产示例：监管者分派 research → draft

无 `DEEPSEEK_API_KEY` 则 SKIP。


In [ ]:
def demo_production_orch() -> None:
    """生产：hops 只有 supervisor↔expert；无 worker 直连。"""
    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP production: DEEPSEEK_API_KEY missing")
        return

    raw = run_supervisor_impl("用三句话介绍 LangGraph 适合什么编排")
    payload = json.loads(raw)
    print("=== supervisor run ===")
    print(json.dumps({"final": payload["final"], "hops": payload["hops"]}, ensure_ascii=False, indent=2)[:1600])
    assert payload["worker_leak"] == []
    dsts = {h["dst"] for h in payload["hops"]}
    assert "research" in dsts or "draft" in dsts
    print("no worker-to-worker leak ok")
    print("PROD DEMO OK")


demo_production_orch()
